In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
# 1. Load Iris Dataset
df = pd.read_csv("Iris.csv")
print(df.head())
print(df.columns)

   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa
Index(['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm',
       'Species'],
      dtype='str')


In [3]:
# 2. Select Features
X = df[
    [
        "SepalLengthCm",
        "SepalWidthCm",
        "PetalLengthCm",
        "PetalWidthCm"
    ]
].values

In [4]:
# 3. Convert Species to Numbers
# Setosa       -> 0
# Versicolor   -> 1
# Virginica    -> 2
species_mapping = {
    "Iris-setosa": 0,
    "Iris-versicolor": 1,
    "Iris-virginica": 2
}

y = df["Species"].map(species_mapping).values

In [5]:
# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
# 5. Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
# 6. Convert Labels to One-Hot Encoding
def one_hot_encode(y, number_of_classes):
    one_hot = np.zeros(
        (len(y), number_of_classes)
    )
    one_hot[np.arange(len(y)), y] = 1
    return one_hot
number_of_classes = 3
y_train_one_hot = one_hot_encode(
    y_train,
    number_of_classes
)

In [8]:
# 7. Softmax Function
def softmax(z):
    # Prevent numerical overflow
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(
        exp_z,
        axis=1,
        keepdims=True
    )

In [9]:
# 8. Multiclass Logistic Regression
class MulticlassLogisticRegression:

    def __init__(
        self,
        learning_rate=0.01,
        epochs=1000
    ):

        self.learning_rate = learning_rate
        self.epochs = epochs

        self.weights = None
        self.bias = None


    def fit(self, X, y):

        number_of_samples = X.shape[0]
        number_of_features = X.shape[1]
        number_of_classes = y.shape[1]


        # Initialize weights and bias

        self.weights = np.zeros(
            (number_of_features, number_of_classes)
        )

        self.bias = np.zeros(
            (1, number_of_classes)
        )


        # Gradient Descent

        for epoch in range(self.epochs):
            # Linear equation
            z = np.dot(X, self.weights) + self.bias

            # Softmax probabilities
            y_predicted = softmax(z)

            # Calculate gradients
            dw = (
                1 / number_of_samples
            ) * np.dot(
                X.T,
                (y_predicted - y)
            )

            db = (
                1 / number_of_samples
            ) * np.sum(
                y_predicted - y,
                axis=0,
                keepdims=True
            )

            # Update parameters
            self.weights -= (
                self.learning_rate * dw
            )

            self.bias -= (
                self.learning_rate * db
            )


    def predict_probability(self, X):

        z = np.dot(
            X,
            self.weights
        ) + self.bias

        return softmax(z)


    def predict(self, X):

        probabilities = self.predict_probability(X)

        return np.argmax(
            probabilities,
            axis=1
        )

In [10]:
# 9. Create Model
model = MulticlassLogisticRegression(
    learning_rate=0.01,
    epochs=1000
)

In [11]:
# 10. Train Model
model.fit(
    X_train,
    y_train_one_hot
)

In [12]:
# 11. Make Predictions
y_pred = model.predict(X_test)

In [13]:
# 12. Accuracy
accuracy = accuracy_score(
    y_test,
    y_pred
)
print("\nAccuracy:")
print(accuracy)


Accuracy:
0.8333333333333334


In [14]:
# 13. Confusion Matrix
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


Confusion Matrix:
[[10  0  0]
 [ 0  6  4]
 [ 0  1  9]]


In [15]:
# 14. Model Parameters
print("\nWeights:")
print(model.weights)
print("\nBias:")
print(model.bias)


Weights:
[[-0.58857279  0.10860563  0.47996716]
 [ 0.62210395 -0.6236242   0.00152026]
 [-0.87175248  0.13809741  0.73365507]
 [-0.8170061  -0.13827545  0.95528155]]

Bias:
[[-0.24304421  0.45967181 -0.2166276 ]]
